# Pricing Project

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from scipy.special import logsumexp

# Load data
candidate_paths = [Path('data.csv'), Path('Data/data.csv')]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find data.csv or Data/data.csv')

df = pd.read_csv(data_path)

# 8 hotel features
feature_cols = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]
required_cols = ['srch_id', 'booking_bool'] + feature_cols
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise KeyError(f'Missing required columns: {missing_cols}')

estimation_df = df.dropna(subset=required_cols).copy()
dropped_rows = len(df) - len(estimation_df)
if dropped_rows > 0:
    print(f'Dropped {dropped_rows} rows with missing required values.')

invalid_searches = estimation_df.groupby('srch_id')['booking_bool'].sum()
if (invalid_searches > 1).any():
    raise ValueError('Each search must have at most one booked hotel.')

# Normalize features so coefficients are comparable across attributes
means = estimation_df[feature_cols].mean()
stds = estimation_df[feature_cols].std()
constant_cols = stds[stds == 0].index.tolist()
if constant_cols:
    raise ValueError(f'Constant features cannot be estimated: {constant_cols}')

norm_cols = [f'{col}_norm' for col in feature_cols]
estimation_df[norm_cols] = (estimation_df[feature_cols] - means) / stds

# Group by search: each search = one customer seeing a set of hotels
search_data = []
for _, group in estimation_df.groupby('srch_id'):
    X = group[norm_cols].to_numpy(dtype=float)
    y = group['booking_bool'].to_numpy(dtype=float)
    search_data.append((X, y))

def neg_log_likelihood_and_grad(beta):
    b0, b = beta[0], beta[1:]
    nll = 0.0
    grad = np.zeros_like(beta)

    for X, y in search_data:
        u = b0 + X @ b
        log_denom = logsumexp(np.concatenate(([0.0], u)))
        probs = np.exp(u - log_denom)
        diff = probs - y

        nll += log_denom - np.dot(y, u)
        grad[0] += diff.sum()
        grad[1:] += X.T @ diff

    return nll, grad

def neg_log_likelihood(beta):
    return neg_log_likelihood_and_grad(beta)[0]

def neg_log_likelihood_grad(beta):
    return neg_log_likelihood_and_grad(beta)[1]

# Optimize
initial_beta = np.zeros(len(feature_cols) + 1)
result = minimize(
    neg_log_likelihood,
    initial_beta,
    jac=neg_log_likelihood_grad,
    method='L-BFGS-B'
)

if not result.success:
    raise RuntimeError(f'Optimization failed: {result.message}')

beta_star = result.x

# Print results
names = ['intercept'] + feature_cols
print(f'Rows used: {len(estimation_df)}')
print(f'Searches used: {estimation_df["srch_id"].nunique()}')
print(f'Bookings: {int(estimation_df["booking_bool"].sum())}')
print(f'No-purchase searches: {int((invalid_searches == 0).sum())}')
print('\nEstimated MNL coefficients:')
for name, val in zip(names, beta_star):
    print(f'  {name:35s} {val:+.6f}')
print(f'\nLog-likelihood: {-result.fun:.4f}')
print(f'Optimizer status: {result.message}')

pretty_names = {
    'prop_starrating': 'star rating',
    'prop_review_score': 'review score',
    'prop_brand_bool': 'brand indicator',
    'prop_location_score': 'location score',
    'prop_accesibility_score': 'accessibility score',
    'prop_log_historical_price': 'historical price',
    'price_usd': 'displayed price',
    'promotion_flag': 'promotion flag'
}

print('\nCoefficient interpretation (holding other features fixed):')
if beta_star[0] < 0:
    print('  intercept: negative, so the outside option is attractive on average.')
else:
    print('  intercept: positive, so booking a displayed hotel is attractive on average.')

for name, val in zip(feature_cols, beta_star[1:]):
    direction = 'raises' if val > 0 else 'lowers'
    print(
        f'  {pretty_names[name]:35s} {direction} hotel utility by {abs(val):.3f} '
        'for a 1 standard deviation increase.'
    )


FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'